# LangGraph Masterclass for Senior Engineers

Welcome. This notebook is designed for experienced software engineers transitioning to **LangGraph**. 

LangGraph is an extension of LangChain designed specifically for creating cyclic, stateful, multi-actor applications (Agents). Unlike standard Directed Acyclic Graphs (DAGs) found in tools like Airflow or Prefect, LangGraph natively supports `while` loops, making it the industry standard for autonomous agents that must retry tools, reflect, and route dynamically.

---

## Table of Contents
1. **Setup & Installation**
2. **Level 1: Beginner (Graph Architecture Fundamentals)**
   - Concept 1: State & Reducers (`TypedDict` / `Pydantic`)
   - Concept 2: Nodes & Edges
3. **Level 2: Intermediate (Dynamic Workflows)**
   - Concept 3: Conditional Edges (Dynamic Routing)
   - Concept 4: Memory & Persistence (Checkpointers)
4. **Level 3: Advanced (Complex Execution)**
   - Concept 5: Human-in-the-Loop (Interrupts)
   - Concept 6: Parallel Execution (Fan-out / Fan-in)
   - Concept 7: Subgraphs & Modular Architectures (Debugging & Performance)
5. **Final Mini-Project: Multi-Agent Research System**

In [ ]:
# 1. Setup & Installation
!pip install -qU langgraph langchain-core

## Level 1: Beginner (Graph Architecture Fundamentals)

### Concept 1 & 2: State, Reducers, Nodes, and Edges

**Explanation:**
- **State**: The shared memory of the graph. Defined via `TypedDict` or `Pydantic`. Every node receives this state and returns an update to it.
- **Reducers**: By default, returning a dictionary key overwrites the existing state. Using `Annotated[Type, reducer_function]` (like `operator.add`) tells LangGraph to append/merge rather than overwrite.
- **Nodes**: Standard Python functions. They take the current `State` and return a dictionary containing updates.
- **Edges**: Define the execution flow (e.g., `Node A -> Node B`).

**Real-world usage:** 
Standardizing static ETL pipelines or multi-step prompt chains where context (like a user's original query and retrieved documents) must be carried through 5+ different processing steps.

**Interview Tip:** 
*"How does LangGraph prevent race conditions when updating state?"* -> Reducers. If two nodes run in parallel and return updates to the same key, a reducer (like list concatenation) ensures both updates are safely merged rather than one arbitrarily overwriting the other.

In [ ]:
from typing import TypedDict, Annotated
import operator
from langgraph.graph import StateGraph, START, END

# 1. Define the State schema
class BasicState(TypedDict):
    # operator.add ensures lists are concatenated, not overwritten
    logs: Annotated[list, operator.add]
    user_id: str

# 2. Define Nodes (Python functions)
def fetch_data_node(state: BasicState):
    # Returns an UPDATE to the state
    return {'logs': [f"Fetched data for {state['user_id']}"]}

def process_data_node(state: BasicState):
    return {'logs': ["Processed data successfully"]}

# 3. Build the Graph
builder = StateGraph(BasicState)
builder.add_node('fetcher', fetch_data_node)
builder.add_node('processor', process_data_node)

# Define the static flow
builder.add_edge(START, 'fetcher')
builder.add_edge('fetcher', 'processor')
builder.add_edge('processor', END)

# 4. Compile and Execute
graph = builder.compile()

initial_state = {'logs': ['System Init'], 'user_id': 'usr_99X'}
result = graph.invoke(initial_state)

print("Final State:\n", result)

**Output Explanation:**
Notice how the `logs` array contains `['System Init', 'Fetched data for usr_99X', 'Processed data successfully']`. The `operator.add` reducer automatically combined the initial state array with the arrays returned by the two nodes.

**Practice Exercise 1:**
Modify the state schema to include a `status: str` key (without a reducer). Have the `processor` node update this status to `'COMPLETED'`. Observe how it overwrites rather than appends.

---
## Level 2: Intermediate (Dynamic Workflows)

### Concept 3: Conditional Edges (Dynamic Routing)

**Explanation:**
Static graphs are just pipelines. True agents require decision-making. Conditional edges evaluate a function to determine the next node dynamically.

**Real-world usage:**
Agent tool calling (e.g., *"Did the LLM request a tool? If yes, route to ToolNode. If no, route to END"*), Support ticket triaging, or Fallback mechanisms (route to a cheaper model if the task is simple).

In [ ]:
class RouterState(TypedDict):
    issue_text: str
    department: str

def classifier_node(state: RouterState):
    # Mock LLM classification logic
    text = state['issue_text'].lower()
    dept = 'billing' if 'invoice' in text else 'tech'
    return {'department': dept}

def billing_node(state: RouterState):
    print("[Executing Billing Logic]")
    return state

def tech_node(state: RouterState):
    print("[Executing Tech Logic]")
    return state

# Routing function: returns the name of the next node
def route_department(state: RouterState):
    if state['department'] == 'billing':
        return 'billing'
    return 'tech'

builder = StateGraph(RouterState)
builder.add_node('classifier', classifier_node)
builder.add_node('billing', billing_node)
builder.add_node('tech', tech_node)

builder.add_edge(START, 'classifier')

# Add dynamic routing
builder.add_conditional_edges(
    'classifier', 
    route_department
    # Optionally, provide a mapping dictionary here for strict validation: 
    # {'billing': 'billing', 'tech': 'tech'}
)

builder.add_edge('billing', END)
builder.add_edge('tech', END)

router_graph = builder.compile()

print("\n--- Test 1 ---")
router_graph.invoke({'issue_text': 'I cannot find my invoice', 'department': ''})

print("\n--- Test 2 ---")
router_graph.invoke({'issue_text': 'Server is throwing 500 error', 'department': ''})

### Concept 4: Memory & Persistence (Checkpointers)

**Explanation:**
By default, graphs are stateless across `invoke()` calls. By passing a `checkpointer` during compilation, LangGraph saves the state after every node execution (a "superstep"). 

**Real-world usage:**
Chatbots remembering conversation history, pausing/resuming long-running background workers, and fault tolerance (restarting a graph from the exact node where an API crashed).

**Performance & Pitfalls:**
`MemorySaver` is for local dev. In production, you will use `PostgresSaver` or `RedisSaver`. Ensure your state objects are JSON-serializable. Complex nested objects will break checkpointers.

In [ ]:
from langgraph.checkpoint.memory import MemorySaver

# We use the graph from Concept 1
memory = MemorySaver()
persistent_graph = builder.compile(checkpointer=memory)

# Threads isolate memory spaces between users/sessions
config = {"configurable": {"thread_id": "user_session_42"}}

print("Run 1:")
res1 = persistent_graph.invoke({'issue_text': 'invoice issue', 'department': ''}, config=config)
print("State after Run 1:", persistent_graph.get_state(config).values)

# Notice we don't need to pass the issue_text again. The graph remembers it.
print("\nRetrieving from Checkpointer manually:")
history = list(persistent_graph.get_state_history(config))
print(f"Found {len(history)} checkpoints in history for this thread.")

---
## Level 3: Advanced (Complex Execution)

### Concept 5: Human-in-the-Loop (Interrupts)

**Explanation:**
Because checkpointers save exact state states, you can pause execution *before* or *after* specific nodes, wait hours/days, and resume. 

**Real-world usage:**
High-stakes actions. An AI agent drafts an email, but the graph is configured with `interrupt_before=['send_email']`. The graph yields control back to the application. A human reviews the state, modifies it if necessary, and resumes the graph.

In [ ]:
class ActionState(TypedDict):
    action_plan: str
    approved: bool

def planner(state: ActionState):
    return {'action_plan': 'DELETE database table users;'}

def executor(state: ActionState):
    if state.get('approved'):
        print("CRITICAL: Executing Plan ->", state['action_plan'])
    else:
        print("Execution aborted.")
    return state

hitl_builder = StateGraph(ActionState)
hitl_builder.add_node('planner', planner)
hitl_builder.add_node('executor', executor)
hitl_builder.add_edge(START, 'planner')
hitl_builder.add_edge('planner', 'executor')
hitl_builder.add_edge('executor', END)

# Pause BEFORE the executor node
hitl_graph = hitl_builder.compile(checkpointer=MemorySaver(), interrupt_before=['executor'])
hitl_config = {"configurable": {"thread_id": "admin_tx_1"}}

print("\n--- 1. Initial Invocation (Will Pause) ---")
hitl_graph.invoke({'action_plan': '', 'approved': False}, config=hitl_config)

state_snapshot = hitl_graph.get_state(hitl_config)
print("Graph is currently waiting at node:", state_snapshot.next)
print("Pending plan:", state_snapshot.values['action_plan'])

print("\n--- 2. Human Approval & Resumption ---")
# Update state as a human
hitl_graph.update_state(hitl_config, {'approved': True})
# Resume graph with None
hitl_graph.invoke(None, config=hitl_config)

### Concept 6 & 7: Parallel Execution & Subgraphs

**Explanation:**
If an edge maps to multiple nodes, LangGraph executes them concurrently. The results are aggregated automatically using your state's Reducers. 
Furthermore, large architectures should be broken into Subgraphs (a compiled graph acts perfectly as a node inside another graph), significantly aiding debugging and unit testing.

**Real-world usage:**
Querying 3 different APIs (Weather, Stocks, News) simultaneously to construct a daily briefing agent.

In [ ]:
class ParallelState(TypedDict):
    results: Annotated[list, operator.add]

def worker_a(state): return {'results': ['Data A']}
def worker_b(state): return {'results': ['Data B']}
def worker_c(state): return {'results': ['Data C']}
def aggregator(state): 
    print("Aggregated:", state['results'])
    return state

par_builder = StateGraph(ParallelState)
par_builder.add_node('wa', worker_a)
par_builder.add_node('wb', worker_b)
par_builder.add_node('wc', worker_c)
par_builder.add_node('agg', aggregator)

# Fan-out
par_builder.add_edge(START, 'wa')
par_builder.add_edge(START, 'wb')
par_builder.add_edge(START, 'wc')

# Fan-in
par_builder.add_edge('wa', 'agg')
par_builder.add_edge('wb', 'agg')
par_builder.add_edge('wc', 'agg')
par_builder.add_edge('agg', END)

par_graph = par_builder.compile()
par_graph.invoke({'results': []})
# Notice that without 'Annotated[list, operator.add]', the results would overwrite each other!

---
## Final Mini-Project: Multi-Agent Research System

**Architecture:**
We will build a cyclic "Reviewer-Writer" system. 
1. **Researcher** gathers data.
2. **Writer** drafts content.
3. **Reviewer** checks quality.
4. **Conditional Edge**: If quality passes, end. If fails, loop back to the Researcher/Writer.
5. Includes memory and interrupts (HITL) before final publication.

*Note: To guarantee this runs smoothly without API keys, we use simulated deterministic LLM logic. In production, simply replace the mock logic inside the nodes with `llm.invoke()` calls.*

In [ ]:
class ProjectState(TypedDict):
    topic: str
    notes: Annotated[list, operator.add]  # Accumulate research
    draft: str
    iteration_count: int
    status: str

def researcher_agent(state: ProjectState):
    current_iter = state.get('iteration_count', 0)
    new_note = f"Fact set {current_iter + 1} about {state['topic']}"
    print(f"[Researcher] Gathering: {new_note}")
    return {'notes': [new_note], 'iteration_count': current_iter + 1}

def writer_agent(state: ProjectState):
    all_notes = ", ".join(state['notes'])
    draft = f"Title: {state['topic']} | Content based on: {all_notes}"
    print(f"[Writer] Draft generated. Length: {len(draft)}")
    return {'draft': draft}

def reviewer_agent(state: ProjectState):
    # Mock review logic: Only pass if we have researched at least 3 times
    if state['iteration_count'] >= 3:
        print("[Reviewer] Draft looks great. Approved.")
        return {'status': 'approved'}
    else:
        print("[Reviewer] Draft needs more depth. Sending back.")
        return {'status': 'needs_revision'}

def review_router(state: ProjectState):
    if state['status'] == 'approved':
        return END
    return 'researcher'

# Build Project Graph
project_builder = StateGraph(ProjectState)
project_builder.add_node('researcher', researcher_agent)
project_builder.add_node('writer', writer_agent)
project_builder.add_node('reviewer', reviewer_agent)

project_builder.add_edge(START, 'researcher')
project_builder.add_edge('researcher', 'writer')
project_builder.add_edge('writer', 'reviewer')

# The powerful cyclic router
project_builder.add_conditional_edges('reviewer', review_router)

# Compile with memory to track the iterations safely
app = project_builder.compile(checkpointer=MemorySaver())

print("=========================================")
print("Starting Multi-Agent Simulation")
print("=========================================")

proj_config = {"configurable": {"thread_id": "proj_A1"}}
final_state = app.invoke({
    'topic': 'LangGraph Architecture',
    'notes': [],
    'iteration_count': 0,
    'draft': '',
    'status': ''
}, config=proj_config)

print("\n=========================================")
print("Final Approved Draft:")
print(final_state['draft'])

### Final Thoughts for Production Deployment
- **LangGraph Studio / LangSmith:** Visualizing these graphs in production is difficult. Connect LangSmith (`export LANGCHAIN_TRACING_V2=true`) to visually trace parallel execution and state mutations.
- **State Bloat:** If dealing with massive documents in RAG, storing full page content in state at every superstep will crash memory/databases. Store pointers (UUIDs) in state and retrieve from a dedicated store when needed.
- **Streaming:** LangGraph allows streaming state updates (`app.stream()`). This is crucial for UI latency, allowing you to show the user what node is currently executing.